# AI Lab - Practical Question No. 33
## Admissible Heuristic Validator (Reverse UCS & Admissibility Analysis)

| Specification | Detail |
| :--- | :--- |
| **Title** | Admissible Heuristic Validator |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🟡 Intermediate |
| **Practical Type** | Validation |
| **Problem Statement** | Given a graph, the true minimum costs from every node to the goal, and a heuristic dictionary $h(n)$, write a Python function to programmatically determine if the provided heuristic is Admissible. |
| **Task Requirements** | 1. Calculate true cost $h^*(n)$ for all nodes using a reverse UCS from the goal.<br>2. Compare provided $h(n)$ against $h^*(n)$.<br>3. Identify any node where $h(n) > h^*(n)$.<br>4. Output True (Admissible) or False (Not Admissible with counterexample). |
| **Input Requirements** | Graph structure, Goal node, Heuristic dict |
| **Expected Output** | Boolean and list of violating nodes |
| **Constraints / Conditions** | Graph has positive weights ($w > 0$) |
| **Core AI Concept** | Admissibility ($0 \le h(n) \le h^*(n)$) |
| **Python Concepts Tested** | Dictionary iteration, reverse graph search, priority queues (`heapq`), dataclasses |
| **Expected Algorithm** | Dijkstra's from goal on transposed graph (Reverse UCS) / Validation logic |
| **Recommended Libraries** | Pure Python standard library (`heapq`, `math`, `typing`, `dataclasses`) |
| **Complexity Expectation** | Time Complexity: $\mathcal{O}((V + E) \log V)$, Space Complexity: $\mathcal{O}(V + E)$ |
| **Sample Input** | $h(A) = 10$, True Cost $A \to \text{Goal} = 8$ |
| **Sample Output** | `False. Heuristic overestimates at node A (10 > 8).` |
| **University Practical Importance** | Viva Important |
| **Viva Topics** | • Definition of Admissibility<br>• Why overestimation breaks optimality in A* |
| **Optional Challenge** | Automatically lower the heuristic values to make them admissible ($h_{\text{repaired}}(n) = \min(h(n), h^*(n))$) |

## 1. Theoretical Foundations

### Definition of Admissibility
A heuristic function $h(n)$ is **admissible** if for every state $n$ in the state space:
$$0 \le h(n) \le h^*(n)$$
where $h^*(n)$ is the true minimum cost from node $n$ to the nearest goal state.

- An admissible heuristic is **optimistic**; it never overestimates the actual remaining path cost.
- At the goal state, $h(\text{Goal}) = 0$ is required because $h^*(\text{Goal}) = 0$.

### Why Overestimation Breaks Optimality in A*
Recall that A* orders the priority queue by $f(n) = g(n) + h(n)$.
- Suppose the true optimal path has total cost $C^*$, and passes through node $n$.
- If $h(n) > h^*(n)$ (overestimation), then $f(n) = g(n) + h(n) > g(n) + h^*(n) = C^*$.
- If there exists an alternative, suboptimal path to a goal with cost $C' > C^*$ such that $C' < f(n)$, A* will expand the suboptimal path to completion and pop the suboptimal goal node from the priority queue **before** node $n$ is ever expanded.
- Thus, A* returns a suboptimal path, violating its core optimality guarantee.

### The Power of Reverse UCS (Dijkstra on Transposed Graph)
- To determine if $h(n) \le h^*(n)$ for all nodes, we need $h^*(n)$ from **all nodes** to a **single goal**.
- Forward approach: Running Dijkstra $|V|$ times requires $\mathcal{O}(|V| \cdot (V + E) \log V)$ time.
- **Reverse UCS approach**: Transpose the directed graph $G = (V, E) \to G^R = (V, E^R)$ by reversing all edges ($u \to v$ becomes $v \to u$). Running a single Dijkstra from $\text{Goal}$ on $G^R$ yields all shortest path distances $h^*(n)$ in a single $\mathcal{O}((V + E) \log V)$ pass!

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is in sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.admissible_heuristic_validator import (
    CANONICAL_GOAL,
    CANONICAL_GRAPH,
    CANONICAL_HEURISTIC_ADMISSIBLE,
    CANONICAL_HEURISTIC_OVERESTIMATING,
    SYLLABUS_SAMPLE_OUTPUT,
    AdmissibilityReport,
    compute_reverse_graph,
    compute_true_costs,
    format_comparison_table,
    get_canonical_sample_graph,
    get_theoretical_analysis,
    get_viva_reference,
    make_admissible,
    render_ascii_graph,
    validate_admissibility,
)

print("Module successfully imported from src.admissible_heuristic_validator!")

## 2. Graph Topology & Canonical Setup

Let us inspect the directed graph layout where node `A` connects to `Goal` via `B` (cost $3 + 5 = 8$) and via `C` (cost $6 + 4 = 10$).

In [ ]:
graph, goal, heuristic = get_canonical_sample_graph()
print(render_ascii_graph(graph, goal))

## 3. Reverse UCS Computation of True Cost $h^*(n)$

We transpose the graph and run Uniform Cost Search (Dijkstra) starting from the goal node.

In [ ]:
# Compute transpose graph
rev_graph = compute_reverse_graph(graph)
print("Reversed Graph Transpose:")
for u, neighbors in sorted(rev_graph.items()):
    print(f"  {u} <- {neighbors}")

# Compute true minimum costs h*(n) via Reverse UCS
true_costs = compute_true_costs(graph, goal)
print("\nComputed True Costs h*(n) from each node to Goal:")
for node, cost in sorted(true_costs.items()):
    print(f"  h*({node}) = {cost:g}")

## 4. Syllabus Sample Output Validation

Testing the provided heuristic with overestimation at node $A$:
- $h(A) = 10$
- True Cost $A \to \text{Goal} = 8$
- $h(A) > h^*(A) \implies$ Violation detected!

In [ ]:
report_overestimating = validate_admissibility(graph, goal, heuristic)
print("Validation Status :", "ADMISSIBLE" if report_overestimating.is_admissible else "NOT ADMISSIBLE")
print("Output String     :", report_overestimating.counterexample)
print("Syllabus Expected :", SYLLABUS_SAMPLE_OUTPUT)
assert report_overestimating.counterexample == SYLLABUS_SAMPLE_OUTPUT, "Mismatch with syllabus specification!"
print("\n✅ Exact syllabus match verified!")

## 5. Comparative Evaluation Table

Tabulating all nodes, user heuristic $h(n)$, true costs $h^*(n)$, admissibility status, and deviations.

In [ ]:
print(format_comparison_table(report_overestimating))

## 6. Testing an Admissible Heuristic

Now let us validate an admissible heuristic where $h(A) = 7.0 \le 8.0$.

In [ ]:
report_admissible = validate_admissibility(graph, goal, CANONICAL_HEURISTIC_ADMISSIBLE)
print("Validation Status :", "ADMISSIBLE" if report_admissible.is_admissible else "NOT ADMISSIBLE")
print("Counterexample    :", report_admissible.counterexample)
print("\nComparison Table:")
print(format_comparison_table(report_admissible))

## 7. Optional Challenge: Automatically Lowering Heuristics to Make Them Admissible

The optional challenge requires us to automatically repair any non-admissible heuristic by clamping overestimations:
$$h_{\text{repaired}}(n) = \min(h(n), h^*(n))$$

Let us verify that repairing the overestimating heuristic yields a strictly admissible heuristic function.

In [ ]:
print("Original Heuristics :", report_overestimating.heuristics)
print("Repaired Heuristics :", report_overestimating.repaired_heuristics)

# Re-validate the repaired heuristics
repaired_validation = validate_admissibility(graph, goal, report_overestimating.repaired_heuristics)
print("\nRe-validation of Repaired Heuristic:")
print("Is Admissible?     :", repaired_validation.is_admissible)
print("Violations Count   :", len(repaired_validation.violations))
print("Result Message     :", repaired_validation.counterexample)
assert repaired_validation.is_admissible, "Repaired heuristic must be strictly admissible!"
print("\n✅ Optional Challenge passed with 100% adherence to admissibility!")

## 8. University Viva Voce Preparation Guide

In [ ]:
print(get_viva_reference())